In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from datetime import timedelta

In [ ]:
file_path = '/content/drive/MyDrive/Hack-o-Week/Week 1/dataset.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print("⚠️ File not found")

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').set_index('Timestamp')
df['Total Load'] = df['Total Load'].interpolate(method='linear')

In [ ]:
df['Smooth Load'] = df['Total Load'].rolling(window=3).mean()

In [ ]:
evening_mask = (df.index.hour >= 18) & (df.index.hour <= 22)
evening_data = df[evening_mask].copy()
daily_peaks = evening_data['Smooth Load'].resample('D').max().dropna().reset_index()
daily_peaks.columns = ['Date', 'Peak Load']
daily_peaks['Day Index'] = (daily_peaks['Date'] - daily_peaks['Date'].min()).dt.days

In [ ]:
train_days = 7
if len(daily_peaks) >= train_days:
    train_data = daily_peaks.tail(train_days)
    X = train_data[['Day Index']]
    y = train_data['Peak Load']
    model = LinearRegression()
    model.fit(X, y)
    last_day_idx = daily_peaks['Day Index'].max()
    next_day_idx = last_day_idx + 1
    predicted_peak = model.predict([[next_day_idx]])[0]
    predicted_date = daily_peaks['Date'].max() + timedelta(days=1)
    print(f"✅ Prediction for {predicted_date.date()} Evening Peak: {predicted_peak:.2f}")
else:
    print("⚠️ Not enough data for 7-day regression.")
    predicted_peak = 0
    predicted_date = daily_peaks['Date'].max() + timedelta(days=1)

✅ Prediction for 2024-02-26 Evening Peak: 137.36


In [ ]:
fig = go.Figure()

# Trace 1: Raw Data (with opacity to show background noise)
fig.add_trace(go.Scatter(
    x=df.index, y=df['Total Load'],
    mode='lines', name='Raw Load',
    line=dict(color='lightgrey', width=1),
    opacity=0.5
))

# Trace 2: Smoothed Moving Average
fig.add_trace(go.Scatter(
    x=df.index, y=df['Smooth Load'],
    mode='lines', name='Smoothed (3H MA)',
    line=dict(color='#1f77b4', width=2)
))

# Trace 3: Daily Evening Peaks
fig.add_trace(go.Scatter(
    x=daily_peaks['Date'] + timedelta(hours=20), # Center marker in evening
    y=daily_peaks['Peak Load'],
    mode='markers', name='Evening Peaks',
    marker=dict(color='red', size=8, symbol='x')
))

# Trace 4: Regression Trend Line (Last 7 Days)
if len(daily_peaks) >= train_days:
    trend_x = [train_data['Date'].min(), predicted_date]
    trend_y_start = model.predict(train_data[['Day Index']].head(1))[0]
    trend_y_end = predicted_peak

    fig.add_trace(go.Scatter(
        x=trend_x, y=[trend_y_start, trend_y_end],
        mode='lines', name='Trend (Last 7 Days)',
        line=dict(color='orange', width=3, dash='dash')
    ))

    # Trace 5: The Prediction Marker
    fig.add_trace(go.Scatter(
        x=[predicted_date], y=[predicted_peak],
        mode='markers+text', name='Next Prediction',
        marker=dict(color='green', size=14, symbol='star'),
        text=[f"{predicted_peak:.1f}"], textposition="top right"
    ))

# Layout with Range Slider for "Live" Timeline feel
fig.update_layout(
    title='Occupancy Load Dashboard: Smoothing & Prediction',
    xaxis_title='Time',
    yaxis_title='Total Load',
    hovermode='x unified',
    template='plotly_white',
    xaxis=dict(
        rangeslider=dict(visible=True), # Adds the slider at bottom
        type='date'
    ),
    height=600
)

# Automatically zoom into the last 14 days for better visibility
last_date = df.index.max()
fig.update_xaxes(range=[last_date - timedelta(days=14), last_date + timedelta(days=1)])

fig.show()